In [1]:
import numpy as np
import torch
import torch.nn as nn
import random
import pandas as pd
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

In [3]:
random.seed(3)
np.random.seed(3)
seed=3
batch_size=1024

In [4]:
device='cuda' if torch.cuda.is_available() else 'cpu'

Census Income是多任务学习和 MMoE 论文中最经典的公开数据集之一,它来源于 美国人口普查（US Census Bureau），目标是根据个人的人口统计和经济属性预测收入及相关社会属性，该数据集有两个任务，一是预测收入是否大于50K，二是预测是否结婚

In [5]:
def load_and_process(seed):
    """
    返回
    train_data        # (N,input_dim)
    train_label       # [income_label(N,), marital_label(N,)]
    validation_data
    validation_label
    test_data
    test_label
    output_info 任务的个数，(每个人物的维度、名称)
    """
    column_names = [
        'age', 'class_worker', 'det_ind_code', 'det_occ_code', 'education',
        'wage_per_hour', 'hs_college', 'marital_stat', 'major_ind_code',
        'major_occ_code', 'race', 'hisp_origin', 'sex', 'union_member',
        'unemp_reason', 'full_or_part_emp', 'capital_gains', 'capital_losses',
        'stock_dividends', 'tax_filer_stat', 'region_prev_res', 'state_prev_res',
        'det_hh_fam_stat', 'det_hh_summ', 'instance_weight', 'mig_chg_msa',
        'mig_chg_reg', 'mig_move_reg', 'mig_same', 'mig_prev_sunbelt',
        'num_emp', 'fam_under_18', 'country_father', 'country_mother',
        'country_self', 'citizenship', 'own_or_self', 'vet_question',
        'vet_benefits', 'weeks_worked', 'year', 'income_50k'
    ]

    label_columns = ['income_50k', 'marital_stat']

    categorical_columns = ['class_worker', 'det_ind_code', 'det_occ_code', 'education', 'hs_college',  'major_ind_code','major_occ_code', 'race', 'hisp_origin', 'sex', 'union_member', 'unemp_reason','full_or_part_emp', 'tax_filer_stat', 'region_prev_res', 'state_prev_res', 'det_hh_fam_stat','det_hh_summ', 'mig_chg_msa', 'mig_chg_reg', 'mig_move_reg', 'mig_same', 'mig_prev_sunbelt','fam_under_18', 'country_father', 'country_mother', 'country_self', 'citizenship','vet_question']

    # 读取数据集
    train_df=pd.read_csv('data/census-income.data.gz',header=None,names=column_names)
    test_df=pd.read_csv('data/census-income.test.gz',header=None,names=column_names)

    # 标签，标签的列名前有空格
    y_train_income=(train_df['income_50k']==' 50000+.').astype(np.int64)
    y_train_marital=(train_df['marital_stat']==' Never married').astype(np.int64)

    y_test_income=(test_df['income_50k']==' 50000+.').astype(np.int64)
    y_test_marital=(test_df['marital_stat']==' Never married').astype(np.int64)

    # 特征
    X_train=train_df.drop(label_columns,axis=1)
    X_test=test_df.drop(label_columns,axis=1)

    # 处理类别特征
    X_all=pd.concat([X_train,X_test],axis=0)
    X_all=pd.get_dummies(X_all,columns=categorical_columns)
    X_all = X_all.astype(np.float32)

    X_train=X_all.iloc[:len(X_train)]
    X_test=X_all.iloc[len(X_train):]

    # 划分验证集和测试集
    X_val, X_test, y_val_income, y_test_income, y_val_marital, y_test_marital = train_test_split(
    X_test, y_test_income, y_test_marital, test_size=0.5, random_state=seed
)
    # val_idx=X_test.sample(frac=0.5,random_state=seed).index
    # test_idx=list(set(X_test.index) - set(val_idx))
    #
    # X_val=X_test.loc[val_idx]
    # X_test=X_test.loc[test_idx]
    #
    # y_val_income=y_test_income.loc[val_idx]
    # y_val_marital=y_test_marital.loc[val_idx]
    #
    # y_test_income=y_test_income.loc[test_idx]
    # y_test_marital=y_test_marital.loc[test_idx]

    return (
        torch.tensor(X_train.values, dtype=torch.float32),
        [torch.tensor(y_train_income.values, dtype=torch.float32),torch.tensor(y_train_marital.values, dtype=torch.float32)],
        torch.tensor(X_val.values, dtype=torch.float32),
        [torch.tensor(y_val_income.values, dtype=torch.float32),torch.tensor(y_val_marital.values, dtype=torch.float32)],
        torch.tensor(X_test.values, dtype=torch.float32),
        [torch.tensor(y_test_income.values, dtype=torch.float32),torch.tensor(y_test_marital.values, dtype=torch.float32)],
        [(2, 'income'), (2, 'marital')]
    )


In [6]:
class censusData(Dataset):
    def __init__(self,train,label):
        super(censusData,self).__init__()
        self.train=train
        self.label=label
    def __getitem__(self,idx):
        x=self.train[idx]
        y=torch.stack([self.label[0][idx],self.label[1][idx]]) # (2,)
        return x,y

    def __len__(self):
        return self.train.shape[0]

In [7]:
class Expert(nn.Module):
    def __init__(self,input_size,output_size,hidden_size):
        super(Expert,self).__init__()
        self.fc1 = nn.Linear(input_size,hidden_size)
        self.fc2 = nn.Linear(hidden_size,output_size)
        self.relu=nn.ReLU()
        self.dropout=nn.Dropout(0.3)
    def forward(self,x):
        out=self.fc1(x)
        out=self.relu(out)
        out=self.dropout(out)
        out=self.fc2(out)
        return out

In [8]:
class Tower(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super(Tower,self).__init__()
        self.fc1 = nn.Linear(input_size,hidden_size)
        self.fc2 = nn.Linear(hidden_size,output_size)
        self.relu=nn.ReLU()
        self.dropout=nn.Dropout(0.4)
        # self.sigmoid=nn.Sigmoid() BCEWithLogitsLoss期望输入的是未归一化的logits
    def forward(self,x):
        out=self.fc1(x)
        out=self.relu(out)
        out=self.dropout(out)
        out=self.fc2(out)
        # out=self.sigmoid(out)
        return out

In [9]:
class MMoE(nn.Module):
    def __init__(self,input_size,num_experts,experts_out,experts_hidden,tower_hidden,tasks):
        super(MMoE,self).__init__()
        self.input_size=input_size
        self.num_experts=num_experts
        self.experts_hidden=experts_hidden
        self.experts_out=experts_out
        self.experts_hidden=experts_hidden
        self.tower_hidden=tower_hidden
        self.tasks=tasks

        self.softmax=nn.Softmax(dim=1)

        self.experts=nn.ModuleList([Expert(self.input_size,self.experts_out,self.experts_out) for _ in range(num_experts)])
        self.w_gates=nn.ParameterList([nn.Parameter(torch.randn(input_size,num_experts),requires_grad=True) for _ in range(tasks)])
        self.towers=nn.ModuleList([Tower(self.experts_out,self.experts_hidden,1) for _ in range(tasks)])
    def forward(self,x):
        """x shape(B,input_size)"""
        experts_o=[e(x) for e in self.experts] # experts*(B,experts_out)
        experts_o_tensor=torch.stack(experts_o,dim=1)  # stack把list中多个tensor沿着新维度拼成一个整体tensor，dim=1是在batch后插入一个expert维度 (B,num_experts,experts_out)

        final_output=[]
        for i in range(len(self.towers)):
            gate_score=self.softmax(torch.matmul(x,self.w_gates[i])) # (B,num_experts)

            #(B,num_experts,1)@(B,num_experts,experts_out) 广播 (B, num_experts, experts_out)求和得到(B,experts_out)
            weighted_expert_out=torch.sum(gate_score.unsqueeze(2) * experts_o_tensor,dim=1)
            final_output.append(self.towers[i](weighted_expert_out)) # (B,1)

        return final_output # tasks *[B,1]

In [10]:
def test(loader,model):
    t1_pred,t2_pred,t1_tag,t2_tag=[],[],[],[]
    model.eval()

    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device)
            yhat=model(x)
            y1,y2=y[:,0],y[:,1]
            yhat_1, yhat_2 = torch.sigmoid(yhat[0]), torch.sigmoid(yhat[1])

            t1_tag.append(y1.cpu().numpy())
            t2_tag.append(y2.cpu().numpy())

            t1_pred.append(yhat_1.detach().cpu().numpy())
            t2_pred.append(yhat_2.detach().cpu().numpy())
    t1_tag=np.concatenate(t1_tag)
    t2_tag=np.concatenate(t2_tag)

    t1_pred=np.concatenate(t1_pred)
    t2_pred=np.concatenate(t2_pred)

    auc_1=roc_auc_score(t1_tag,t1_pred)
    auc_2=roc_auc_score(t2_tag,t2_pred)
    return auc_1,auc_2

In [11]:
train_data, train_label, validation_data, validation_label, test_data, test_label, output_info = load_and_process(seed)

In [12]:
train_loader=DataLoader(censusData(train_data,train_label),batch_size=batch_size,shuffle=True)
val_loader=DataLoader(censusData(validation_data,validation_label),batch_size=batch_size,shuffle=True)
test_loader=DataLoader(censusData(test_data,test_label),batch_size=batch_size,shuffle=True)

In [13]:
model=MMoE(input_size=499,num_experts=6,experts_out=16,experts_hidden=32,tower_hidden=8,tasks=2)
model=model.to(device)

In [14]:
lr=1e-4
n_epochs=80
loss_fn = nn.BCEWithLogitsLoss()
optimizer=optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5)
losses=[]
val_loss=[]

In [15]:
for epoch in tqdm(range(1,n_epochs+1)):
    model.train()
    epoch_loss=[]

    for x,y in train_loader:
        x,y=x.to(device),y.to(device)
        y_hat=model(x)

        y1,y2=y[:,0],y[:,1] # 真实标签
        y_1,y_2=y_hat[0],y_hat[1] # 预测输出

        loss1,loss2=loss_fn(y_1,y1.view(-1,1)),loss_fn(y_2,y2.view(-1,1))
        loss=loss1+loss2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss.append(loss.item())
    losses.append(np.mean(epoch_loss))

    auc1,auc2=test(val_loader,model)
    if epoch==1 or epoch%10==0:print(f'epoch: {epoch}, train loss: {np.mean(epoch_loss)} val task1 auc: {auc1:.5f}, val task2 auc: {auc2:.3f}')

auc1,auc2=test(test_loader,model)
print(f'test auc1: {auc1:.3f}, test auc2: {auc2:.3f}')

  1%|▏         | 1/80 [00:02<03:51,  2.93s/it]

epoch: 1, train loss: 2.07867484887441 val task1 auc: 0.74782, val task2 auc: 0.641


 12%|█▎        | 10/80 [00:27<03:11,  2.73s/it]

epoch: 10, train loss: 0.5348613558671413 val task1 auc: 0.89748, val task2 auc: 0.957


 25%|██▌       | 20/80 [00:54<02:44,  2.73s/it]

...

 88%|████████▊ | 70/80 [03:13<00:28,  2.82s/it]

epoch: 70, train loss: 0.2611252230711472 val task1 auc: 0.94478, val task2 auc: 0.993


100%|██████████| 80/80 [03:40<00:00,  2.76s/it]

epoch: 80, train loss: 0.25530913884823137 val task1 auc: 0.94515, val task2 auc: 0.993


test auc1: 0.946, test auc2: 0.993


论文 income:0.941 marital:0.9927
实现 income: 0.946 marital:0.993